Install the necessary packages

In [ ]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.2/36.2 MB 55.5 MB/s eta 0:00:00


In [ ]:
!git clone https://github.com/t7morgen/misato-dataset.git

Cloning into 'misato-dataset'...
remote: Enumerating objects: 274, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (130/130), done.
remote: Total 274 (delta 90), reused 90 (delta 38), pack-reused 105 (from 1)
Receiving objects: 100% (274/274), 174.84 MiB | 28.70 MiB/s, done.
Resolving deltas: 100% (97/97), done.
Updating files: 100% (91/91), done.


Download the PDB files

In [ ]:
import requests
import os

def download_pdb(pdb_id, output_dir='pdb_files'):
    """Download PDB file from RCSB PDB"""
    os.makedirs(output_dir, exist_ok=True)

    # PDB download URL
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"

    response = requests.get(url)
    if response.status_code == 200:
        filepath = os.path.join(output_dir, f"{pdb_id}.pdb")
        with open(filepath, 'w') as f:
            f.write(response.text)
        print(f"Downloaded {pdb_id}.pdb")
        return filepath
    else:
        print(f"Failed to download {pdb_id}")
        return None

# Example usage
pdb_ids = ['10GS', '11GS', '13GS', '16PK', '184L', '185L', '186L', '187L', '188L', '1A07', '1A08', '1A09', '1A0Q', '1A1B', '1A1C', '1A1E', '1A28', '1A2C', '1A30', '1A3E']  # Your MISATO PDB IDs
for pdb_id in pdb_ids:
    download_pdb(pdb_id)

Downloaded 10GS.pdb
Downloaded 11GS.pdb
Downloaded 13GS.pdb
Downloaded 16PK.pdb
Downloaded 184L.pdb
Downloaded 185L.pdb
Downloaded 186L.pdb
Downloaded 187L.pdb
Downloaded 188L.pdb
Downloaded 1A07.pdb
Downloaded 1A08.pdb
Downloaded 1A09.pdb
Downloaded 1A0Q.pdb
Downloaded 1A1B.pdb
Downloaded 1A1C.pdb
Downloaded 1A1E.pdb
Downloaded 1A28.pdb
Downloaded 1A2C.pdb
Downloaded 1A30.pdb
Downloaded 1A3E.pdb


Decide the protein for which you want to compare its PDB file and Misato file

In [ ]:
protein = '16PK'

Create the corresponding pdb file from Misato

In [ ]:
%cd /content/misato-dataset/src/data/processing

!python3 h5_to_pdb.py --struct $protein --datasetMD /content/misato-dataset/data/MD/h5_files/tiny_md.hdf5

/content/misato-dataset/src/data/processing
Generating pdb for MD dataset for 16PK frame 0


Decide which molecule you want to compare the atoms of:

Each pdb file can have multiple molecules which are distinguished by **chainid**. Misato files also have multiple molecules distinguised by **molecules_begin_atom_index**.

Here, we only compare atoms per molecule. Please modify the **pdb_molecule_chainid** and **misato_molecule_count** variables to decide the molecule.

In [ ]:
import h5py
import numpy as np

mddata = h5py.File('/content/misato-dataset/data/MD/h5_files/tiny_md_out.hdf5')
pr = mddata[protein]

pr['molecules_begin_atom_index'][()]

array([   0, 3127])

In [ ]:
# Which PDB and Misato molecule?
pdb_molecule_chainid = 'A'
misato_molecule_count = 3127

For PDB: Update dictmolpdb with details of Atoms in the decided molecule

In [ ]:
from rdkit import Chem

mol1 = Chem.MolFromPDBFile(f'/content/pdb_files/{protein}.pdb', removeHs=True)

unique_atoms1 = set()

dictmolpdb = {}

for atom1 in mol1.GetAtoms():
    pdb_info1 = atom1.GetMonomerInfo()

    atom_name1 = pdb_info1.GetName().strip()  # This is the locant
    residue_name1 = pdb_info1.GetResidueName()
    residue_number1 = pdb_info1.GetResidueNumber()
    chainid1 = pdb_info1.GetChainId()

    unique_atoms1.add(atom_name1)
    if chainid1 == pdb_molecule_chainid:
      if residue_number1 not in dictmolpdb:
        dictmolpdb[residue_number1] = [atom_name1]
      else:
        dictmolpdb[residue_number1].append(atom_name1)

    print(chainid1, residue_number1, residue_name1, atom_name1)

A 5 GLU N
A 5 GLU CA
A 5 GLU C
A 5 GLU O
A 5 GLU CB
A 5 GLU CG
A 5 GLU CD
A 5 GLU OE1
A 5 GLU OE2
A 6 LYS N
A 6 LYS CA
A 6 LYS C
A 6 LYS O
A 6 LYS CB
A 6 LYS CG
A 6 LYS CD
A 6 LYS CE
A 6 LYS NZ
A 7 LYS N
A 7 LYS CA
A 7 LYS C
A 7 LYS O
A 7 LYS CB
A 7 LYS CG
A 7 LYS CD
A 7 LYS CE
A 7 LYS NZ
A 8 SER N
A 8 SER CA
A 8 SER C
A 8 SER O
A 8 SER CB
A 8 SER OG
A 9 ILE N
A 9 ILE CA
A 9 ILE C
A 9 ILE O
A 9 ILE CB
A 9 ILE CG1
A 9 ILE CG2
A 9 ILE CD1
A 10 ASN N
A 10 ASN CA
A 10 ASN C
A 10 ASN O
A 10 ASN CB
A 10 ASN CG
A 10 ASN OD1
A 10 ASN ND2
A 11 GLU N
A 11 GLU CA
A 11 GLU C
A 11 GLU O
A 11 GLU CB
A 11 GLU CG
A 11 GLU CD
A 11 GLU OE1
A 11 GLU OE2
A 12 CYS N
A 12 CYS CA
A 12 CYS C
A 12 CYS O
A 12 CYS CB
A 12 CYS SG
A 13 ASP N
A 13 ASP CA
A 13 ASP C
A 13 ASP O
A 13 ASP CB
A 13 ASP CG
A 13 ASP OD1
A 13 ASP OD2
A 14 LEU N
A 14 LEU CA
A 14 LEU C
A 14 LEU O
A 14 LEU CB
A 14 LEU CG
A 14 LEU CD1
A 14 LEU CD2
A 15 LYS N
A 15 LYS CA
A 15 LYS C
A 15 LYS O
A 15 LYS CB
A 15 LYS CG
A 15 LYS CD
A 15 LYS CE
A 15 

For MISATO: Update dictmolmisato with details of Atoms in the decided molecule

In [ ]:
mol2 = Chem.MolFromPDBFile('/content/misato-dataset/src/data/processing/16PK_MD_frame0.pdb', removeHs=True)

unique_atoms2 = set()

dictmolmisato = {}

count = 0

for atom2, atomtype in zip(mol2.GetAtoms(), pr['atoms_type'][()]):
    pdb_info2 = atom2.GetMonomerInfo()

    atom_name2 = pdb_info2.GetName().strip()  # This is the locant
    residue_name2 = pdb_info2.GetResidueName()
    residue_number2 = pdb_info2.GetResidueNumber()

    unique_atoms2.add(atom_name2)

    if count < misato_molecule_count:
      if residue_number2 not in dictmolmisato:
        dictmolmisato[residue_number2] = [atom_name2]
      else:
        dictmolmisato[residue_number2].append(atom_name2)

    print(residue_number1, residue_name1, atom_name1)

    count += 1

print(count)

1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O
1079 HOH O

[14:50:37] WARNING: not removing hydrogen atom without neighbors
[14:50:37] WARNING: not removing hydrogen atom without neighbors
[14:50:37] WARNING: not removing hydrogen atom without neighbors
[14:50:37] WARNING: not removing hydrogen atom without neighbors
[14:50:37] WARNING: not removing hydrogen atom without neighbors
[14:50:37] WARNING: not removing hydrogen atom without neighbors
[14:50:37] WARNING: not removing hydrogen atom without neighbors
[14:50:37] WARNING: not removing hydrogen atom without neighbors
[14:50:37] WARNING: not removing hydrogen atom without neighbors
[14:50:37] WARNING: not removing hydrogen atom without neighbors
[14:50:37] WARNING: not removing hydrogen atom without neighbors
[14:50:37] WARNING: not removing hydrogen atom without neighbors
[14:50:37] WARNING: not removing hydrogen atom without neighbors
[14:50:37] WARNING: not removing hydrogen atom without neighbors
[14:50:37] WARNING: not removing hydrogen atom without neighbors
[14:50:37] WARNING: not r

Compare dictmolpdb and dictmolmisato

In [ ]:
print(len(dictmolpdb), len(dictmolmisato))
for i,j in zip(dictmolpdb, dictmolmisato):
  if sorted(dictmolpdb[i]) != sorted(dictmolmisato[j]):
    print(i, sorted(dictmolpdb[i]))
    print(j, sorted(dictmolmisato[j]))

997 415
5 ['C', 'CA', 'CB', 'CD', 'CG', 'N', 'O', 'OE1', 'OE2']
1 ['C', 'CA', 'CB', 'CD', 'CG', 'N', 'O', 'O', 'OE1']


In the Misato paper, they have mentioned that they use the **pdbbind pdb files**. Hence, we can download the pdbbind pdb files from 'Protein-ligand complex structures' in https://www.pdbbind-plus.org.cn/download

After downloading, **place the proteinname_protein.pdb in the colab folder, modify the path of mol1 and rerun all the cells**.

mol1 = Chem.MolFromPDBFile(f'/content/pdb_files/{protein}.pdb', removeHs=True)